# 🚗 Project 4.4 — CAN Bus Log Merger
**DSA for Mechatronics · Week 04 — Linked Lists**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
Two CAN bus channels (A and B) each produce a log of messages sorted by timestamp.
After a test drive they must be **merged** into one chronological log for analysis.
Occasionally a logging bug creates a **cycle** in one of the raw buffers (a pointer loops back) — this must be detected and reported before merging.


## Step 1 — Generate two sorted CAN message logs

In [ ]:
class MNode:
    """One CAN message in the log."""
    def __init__(self, ts, can_id, data_byte, channel):
        self.ts        = ts           # timestamp in ms
        self.can_id    = can_id       # 11-bit CAN ID (0x000–0x7FF)
        self.data_byte = data_byte    # a single payload byte
        self.channel   = channel      # 'A' or 'B'
        self.next      = None
    def __repr__(self):
        return f"[{self.ts}ms 0x{self.can_id:03X} ch={self.channel}]"

def build_sorted_log(channel, n, ts_start, ts_end):
    """Build a sorted (by timestamp) linked list of CAN messages."""
    timestamps = sorted(random.sample(range(ts_start, ts_end), n))
    head = tail = None
    for ts in timestamps:
        node = MNode(
            ts        = ts,
            can_id    = random.randint(0x100, 0x7FF),
            data_byte = random.randint(0, 255),
            channel   = channel,
        )
        if head is None:
            head = tail = node
        else:
            tail.next = node
            tail = node
    return head, tail

# ── Personalised parameters ────────────────────────────────────────
N_A       = random.randint(12, 20)
N_B       = random.randint(10, 18)
TS_RANGE  = random.randint(800, 1500)

head_a, tail_a = build_sorted_log("A", N_A, 0, TS_RANGE)
head_b, tail_b = build_sorted_log("B", N_B, 0, TS_RANGE)

def log_to_list(h):
    out, cur = [], h
    while cur:
        out.append(cur)
        cur = cur.next
    return out

msgs_a = log_to_list(head_a)
msgs_b = log_to_list(head_b)

print(f"CAN bus log parameters:")
print(f"  Channel A messages : {N_A}")
print(f"  Channel B messages : {N_B}")
print(f"  Timestamp range    : 0 – {TS_RANGE} ms")
print()
print("Channel A (first 6):")
for m in msgs_a[:6]:
    print(f"  {m.ts:5d} ms  0x{m.can_id:03X}  data=0x{m.data_byte:02X}")
print("Channel B (first 6):")
for m in msgs_b[:6]:
    print(f"  {m.ts:5d} ms  0x{m.can_id:03X}  data=0x{m.data_byte:02X}")


## Step 2 — Cycle detection with Floyd's algorithm

In [ ]:
def has_cycle(head):
    """
    Floyd's cycle detection: fast pointer moves 2 steps, slow moves 1.
    If they ever meet, there is a cycle. O(n) time, O(1) space.
    """
    if not head: return False
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

def find_cycle_entry(head):
    """
    After detecting a cycle, find the entry node.
    Phase 1: detect meeting point.
    Phase 2: reset one pointer to head; both advance 1 step — they meet at entry.
    """
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            # cycle exists — find entry
            slow = head
            while slow is not fast:
                slow = slow.next
                fast = fast.next
            return slow
    return None

# Inject a cycle into a copy of log A for demonstration (with low probability)
inject_cycle = random.random() < 0.5
cycle_info   = "none"

if inject_cycle and len(msgs_a) >= 4:
    # Create a copy list to test cycle detection (don't corrupt real log)
    test_head = MNode(0, 0x000, 0, "TEST")
    test_cur  = test_head
    cycle_target = None
    for i, m in enumerate(msgs_a):
        node = MNode(m.ts, m.can_id, m.data_byte, m.channel)
        test_cur.next = node
        test_cur = node
        if i == len(msgs_a) // 3:
            cycle_target = node     # remember this node
    test_cur.next = cycle_target    # close the cycle
    test_head = test_head.next      # skip dummy

    cycle_detected = has_cycle(test_head)
    entry = find_cycle_entry(test_head)
    cycle_info = f"detected at ts={entry.ts}ms" if entry else "detected (entry unknown)"
else:
    # No cycle injected — verify clean logs
    cycle_detected = has_cycle(head_a) or has_cycle(head_b)
    cycle_info = "none detected in real logs"

print(f"Cycle detection (Floyd's algorithm):")
print(f"  Cycle injected     : {inject_cycle}")
print(f"  Cycle detected     : {cycle_detected}")
print(f"  Cycle info         : {cycle_info}")
print(f"  Real log A clean   : {not has_cycle(head_a)}")
print(f"  Real log B clean   : {not has_cycle(head_b)}")


## Step 3 — Merge two sorted lists (dummy head + two-pointer)

In [ ]:
def merge_sorted_logs(l1, l2):
    """
    Merge two sorted linked lists into one sorted list.
    Uses dummy head to avoid special-casing the first node.
    Two-pointer: always pick the smaller timestamp.
    O(n+m) time, O(1) extra space (reuses existing nodes).
    """
    dummy   = MNode(-1, 0, 0, "-")
    current = dummy

    p, q = l1, l2
    while p and q:
        if p.ts <= q.ts:
            current.next = p
            p = p.next
        else:
            current.next = q
            q = q.next
        current = current.next

    current.next = p if p else q   # attach remaining
    return dummy.next

merged_head = merge_sorted_logs(head_a, head_b)
merged      = log_to_list(merged_head)

# verify sorted
is_sorted = all(merged[i].ts <= merged[i+1].ts for i in range(len(merged)-1))

print(f"Merged log ({N_A + N_B} messages, sorted: {is_sorted}):")
print()
print(f"  {'#':>4}  {'Time':>7}  {'Ch':>4}  {'CAN ID':>7}  {'Data':>6}")
print(f"  {'─'*4}  {'─'*7}  {'─'*4}  {'─'*7}  {'─'*6}")
for i, m in enumerate(merged[:14]):
    print(f"  {i:4d}  {m.ts:5d}ms  {m.channel:>4}  0x{m.can_id:03X}    0x{m.data_byte:02X}")
if len(merged) > 14:
    print(f"  ... and {len(merged)-14} more")


## Step 4 — Analyse the merged log: busiest window (prefix sum on linked list)

In [ ]:
def busiest_window(head_node, window_ms):
    """
    Find the time window of `window_ms` milliseconds with the most messages.
    Sliding two-pointer on the sorted linked list.
    O(n) time, O(1) space.
    """
    if not head_node: return 0, 0, 0

    left  = head_node
    right = head_node
    count = 1
    best_count = 1
    best_start = head_node.ts

    while right.next:
        right = right.next
        count += 1
        # shrink left if window exceeded
        while right.ts - left.ts > window_ms:
            left  = left.next
            count -= 1
        if count > best_count:
            best_count = count
            best_start = left.ts

    return best_count, best_start, best_start + window_ms

WINDOW_MS = random.randint(80, 150)
best_cnt, win_start, win_end = busiest_window(merged_head, WINDOW_MS)

# also count per-channel
a_count = sum(1 for m in merged if m.channel == "A")
b_count = sum(1 for m in merged if m.channel == "B")

print(f"Busiest {WINDOW_MS}ms window: {best_cnt} messages  ({win_start}ms – {win_end}ms)")
print()
print(f"Channel breakdown:")
print(f"  Channel A: {a_count} messages  ({a_count/(N_A+N_B)*100:.1f}%)")
print(f"  Channel B: {b_count} messages  ({b_count/(N_A+N_B)*100:.1f}%)")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " CAN BUS LOG MERGER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  LOG PARAMETERS" + " "*(W-16) + "║")
print(f"║  {'Channel A messages':<28}: {N_A:<26}║")
print(f"║  {'Channel B messages':<28}: {N_B:<26}║")
print(f"║  {'Timestamp range':<28}: 0 – {TS_RANGE} ms{'':<20}║")
print("╠" + "═"*W + "╣")
print("║  CYCLE DETECTION (FLOYD)" + " "*(W-25) + "║")
print(f"║  {'Cycle injected into test':<28}: {str(inject_cycle):<26}║")
print(f"║  {'Cycle detected':<28}: {str(cycle_detected):<26}║")
print(f"║  {'Result':<28}: {cycle_info:<26}║")
print(f"║  {'Real logs safe to merge':<28}: {str(not has_cycle(merged_head)):<26}║")
print("╠" + "═"*W + "╣")
print("║  MERGE RESULTS" + " "*(W-15) + "║")
print(f"║  {'Total messages merged':<28}: {N_A + N_B:<26}║")
print(f"║  {'Chronologically sorted':<28}: {str(is_sorted):<26}║")
print(f"║  {'Channel A share':<28}: {a_count} msgs ({a_count/(N_A+N_B)*100:.1f}%){'':<13}║")
print(f"║  {'Channel B share':<28}: {b_count} msgs ({b_count/(N_A+N_B)*100:.1f}%){'':<13}║")
print("╠" + "═"*W + "╣")
print("║  TRAFFIC ANALYSIS" + " "*(W-18) + "║")
print(f"║  {'Analysis window':<28}: {WINDOW_MS} ms{'':<23}║")
print(f"║  {'Peak messages in window':<28}: {best_cnt:<26}║")
print(f"║  {'Peak window':<28}: {win_start}ms – {win_end}ms{'':<16}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which linked list concept did you find most important, and why?**
Pick one technique from the notebook (fast/slow pointers, dummy head, reversal, cycle detection…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
